In [ ]:
###########################################################
# 과 제 명: [A2024_S05_2] 생활인구 분석 - 경기 부천시[31050]
# 파 일 명: S2.1.유동인구_DATA.ipynb
# 작 성 자: 한영기
###########################################################
import os, importlib, warnings; warnings.filterwarnings(action='ignore')
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from config import config_A28       as config   ; importlib.reload(config)
from utils  import functions_utils    as fu       ; importlib.reload(fu)
from utils  import functions_log      as fl       ; importlib.reload(fl)
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from utils  import functions_lime     as flime    ; importlib.reload(flime)
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from utils  import functions_utils_db as fud      ; importlib.reload(fud)

from shapely          import wkt #함수는 문자열로 표현된 지오메트리를 파싱하여 실제 지오메트리 객체로 변환
from shapely.geometry import Polygon, mapping, shape,Point
import geopandas as gpd
#==================================================================================================
# [0-1] log 기록 정의
log = fl.Logger(config.PATH_LOG, config.LOG_FILE_NM ); log.set_log_level("info")
log.info("0-1 log 기록 정의")
#==================================================================================================
# [0-2] 실행 변수 정의
log.info("0.2.실행 변수 정의")
config_info = fu.f_get_config_info(config.config_run, config.ENV_DEV)
#==================================================================================================
# [0-3] 실행 변수
log.info("0.3.실행 변수 추가"); fu.f_memory_check()
# 전역 변수 선언 부분
# config_info["TODAY"]           = "20230630"

In [ ]:
# [분석 처리 시작]
###########################################################
# 모 듈 명: 모 듈 명 - S2.1.유동인구_DATA.ipynb
# 파 일 명: S2.1.유동인구_DATA.ipynb
# 작 성 자: 한영기
# 설    명: 설명
# 파일 버전: v1.0 [2024.07.22]
# 작성 일자: 2024.07.22
# 수정 일자: 2024.07.22 00:00 최초 작성
###########################################################
fl.f_log_line('[START]S2.1.유동인구_DATA.ipynb')
# 1.실행 변수 정의
log.info("1.실행 변수 정의")
path_base:    str = "2024-06-14"
V_DATE_START: str = '2023-01-01'
V_DATE_END:   str = '2024-05-31'

path_data_out     = os.path.join(config_info["PATH_DATA_COLLECT"],path_base,"01.외부")
path_data_in      = os.path.join(config_info["PATH_DATA_COLLECT"],path_base,"02.내부")
path_data_analyze = os.path.join(config_info["PATH_DATA_ANALYZE"])

path_database = os.path.join(config_info["PATH_DATA_OUTPUT_DATEBASE"],path_base)
path_pretreat = os.path.join(config_info["PATH_DATA_OUTPUT_PRETREAT"],path_base)
path_middle   = os.path.join(config_info["PATH_DATA_OUTPUT_MIDDLE"]  ,path_base)
path_result   = os.path.join(config_info["PATH_DATA_OUTPUT_RESULT"]  ,path_base)

---

In [ ]:
###########################################################################
# [2.1][DB][e10_유동인구_flowpop] ← [d10_유동인구_flowpop]
###########################################################################
# [GG_50CELL_FLOWPOP][유동인구_셀 단위 집계]
# 1.원천 데이터 정보 : [d10_유동인구_flowpop]
# 2.타켓 데이터 정보 : [e10_유동인구_flowpop]
###########################################################################
#==========================================================================
log_text = ["2.1","[e10_유동인구_flowpop]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 1.출력정보 저장처리
config_info["p_gubun"]      = "db" # 저장 형태 [csv, xls, xlsx, db, txt, shp]
config_info["p_table_name"] = "e10_유동인구_flowpop"
config_info["p_if_exists"]  = "replace" #[replace][append]
#==========================================================================
p_sql = f"""
-- [e10_유동인구_flowpop] <- [d10_유동인구_flowpop]
with t_1 as (
    select
           ETL_YMD   as 년월일
          ,ADMI_CD   as 행정코드
          ,CTY_NM    as 시군구명
          ,ADMI_NM   as 행정동명
--          ,UTMK_X
--          ,UTMK_Y
          ,TIME_CD   as 시간구분
--           ,FORN_GB
          ,sum(M_10_CNT) as M_10_CNT
          ,sum(M_15_CNT) as M_15_CNT
          ,sum(M_20_CNT) as M_20_CNT
          ,sum(M_25_CNT) as M_25_CNT
          ,sum(M_30_CNT) as M_30_CNT
          ,sum(M_35_CNT) as M_35_CNT
          ,sum(M_40_CNT) as M_40_CNT
          ,sum(M_45_CNT) as M_45_CNT
          ,sum(M_50_CNT) as M_50_CNT
          ,sum(M_55_CNT) as M_55_CNT
          ,sum(M_60_CNT) as M_60_CNT
          ,sum(M_65_CNT) as M_65_CNT
          ,sum(M_70_CNT) as M_70_CNT
          ,sum(F_10_CNT) as F_10_CNT
          ,sum(F_15_CNT) as F_15_CNT
          ,sum(F_20_CNT) as F_20_CNT
          ,sum(F_25_CNT) as F_25_CNT
          ,sum(F_30_CNT) as F_30_CNT
          ,sum(F_35_CNT) as F_35_CNT
          ,sum(F_40_CNT) as F_40_CNT
          ,sum(F_45_CNT) as F_45_CNT
          ,sum(F_50_CNT) as F_50_CNT
          ,sum(F_55_CNT) as F_55_CNT
          ,sum(F_60_CNT) as F_60_CNT
          ,sum(F_65_CNT) as F_65_CNT
          ,sum(F_70_CNT) as F_70_CNT
--           ,X
--           ,Y
      from d10_유동인구_flowpop a
     group by
           ETL_YMD
          ,ADMI_NM
          ,ADMI_CD
          ,CTY_NM
          ,TIME_CD
-- )select * from t_1 b; /* [] */
)
    select
           년월일
          ,행정코드
          ,시군구명
          ,행정동명
          ,시간구분
          ,M_10_CNT
          ,M_15_CNT
          ,M_20_CNT
          ,M_25_CNT
          ,M_30_CNT
          ,M_35_CNT
          ,M_40_CNT
          ,M_45_CNT
          ,M_50_CNT
          ,M_55_CNT
          ,M_60_CNT
          ,M_65_CNT
          ,M_70_CNT
          ,F_10_CNT
          ,F_15_CNT
          ,F_20_CNT
          ,F_25_CNT
          ,F_30_CNT
          ,F_35_CNT
          ,F_40_CNT
          ,F_45_CNT
          ,F_50_CNT
          ,F_55_CNT
          ,F_60_CNT
          ,F_65_CNT
          ,F_70_CNT
          ,CASE DAYOFWEEK(STR_TO_DATE(년월일, '%Y%m%d'))
                WHEN 1 THEN '일'
                WHEN 2 THEN '월'
                WHEN 3 THEN '화'
                WHEN 4 THEN '수'
                WHEN 5 THEN '목'
                WHEN 6 THEN '금'
                WHEN 7 THEN '토'
            END AS 요일구분
      from t_1
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    log.info(f"{log_text[0]}.2.SQL 실행결과 가져오기 ")
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f"{log_text[0]}.3.테이블 DROP")
    cur.execute(f'DROP table IF EXISTS {config_info["p_table_name"]}')
    log.info(f"{log_text[0]}.4.결과저장처리")
    result = flime.f_data_save2(p_log=log, p_data_object=df, p_config_info=config_info)

    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 년월일 VARCHAR(8) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 행정동명 VARCHAR(20) DEFAULT NULL NULL')
    cur.execute(f'CREATE INDEX {config_info["p_table_name"]}_년월일_IDX ON {config_info["p_table_name"]} (년월일,행정동명)')

    log.info(f"{log_text[0]}.[종료]{log_text[1]} [추출건수: {len(df)}] [처리시간:{fu.f_duration_time(p_s_time)}]")
except Exception as e:
    log.error(f'      file 예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/23 17:53:52] 2.1.[종료][e10_유동인구_flowpop] [추출건수: 42408] [처리시간:0:03:00]
#[INFO   ][2024/07/24 10:04:01] 2.1.[종료][e10_유동인구_flowpop] [추출건수: 282400] [처리시간:0:29:35]

In [ ]:
###########################################################################
# [2.11][DB][e11_유동인구_flowpop] ← [e10_유동인구_flowpop]
###########################################################################
# [GG_50CELL_FLOWPOP][유동인구_셀 단위 집계]
# 1.원천 데이터 정보 : [e10_유동인구_flowpop]
# 2.타켓 데이터 정보 : [e11_유동인구_flowpop]
###########################################################################
#==========================================================================
log_text = ["2.11","[e11_유동인구_flowpop]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 1.출력정보 저장처리
config_info["p_gubun"]      = "db" # 저장 형태 [csv, xls, xlsx, db, txt, shp]
config_info["p_table_name"] = "e11_유동인구_flowpop"
config_info["p_if_exists"]  = "replace" #[replace][append]
#==========================================================================
p_sql = f"""
-- [e11_유동인구_flowpop] <- [e10_유동인구_flowpop]
WITH RECURSIVE months AS (
    SELECT DATE('{V_DATE_START}') AS first_day UNION ALL
    SELECT DATE_ADD(first_day, INTERVAL 1 MONTH) FROM months
     WHERE DATE_ADD(first_day, INTERVAL 1 MONTH) <= DATE('{V_DATE_END}')
), month_day_cnt as (
    SELECT DATE_FORMAT(first_day, '%Y%m') AS 년월
          ,DAY(LAST_DAY(first_day))       AS days_in_month
    from months
-- ) select * from month_day_cnt;
), month_cnt as (
    SELECT substring(년월,1,4)  AS 년 ,count(distinct 년월) AS cnt_month
      from month_day_cnt group by substring(년월,1,4)
-- ) select * from month_cnt;
), a_1 as (
    select distinct 맵핑_T as 남여구분명 from t00_기준코드 where 구분 = '남여구분'
-- )select * from a_1;
), a_2 as (
    select distinct 맵핑_T as 연령구분명 from t00_기준코드 where 구분 = '연령대_kosis'
-- )select * from a_2;
), t_1 as (
    select
           substring(년월일,1,6) as 년월
          ,행정코드
          ,시군구명
          ,행정동명
          ,요일구분
          ,시간구분
          ,round(sum(M_10_CNT+M_15_CNT),0) as m_10
          ,round(sum(M_20_CNT+M_25_CNT),0) as m_20
          ,round(sum(M_30_CNT+M_35_CNT),0) as m_30
          ,round(sum(M_40_CNT+M_45_CNT),0) as m_40
          ,round(sum(M_50_CNT+M_55_CNT),0) as m_50
          ,round(sum(M_60_CNT+M_65_CNT),0) as m_60
          ,round(sum(M_70_CNT),0)          as m_70
          ,round(sum(F_10_CNT+F_15_CNT),0) as w_10
          ,round(sum(F_20_CNT+F_25_CNT),0) as w_20
          ,round(sum(F_30_CNT+F_35_CNT),0) as w_30
          ,round(sum(F_40_CNT+F_45_CNT),0) as w_40
          ,round(sum(F_50_CNT+F_55_CNT),0) as w_50
          ,round(sum(F_60_CNT+F_65_CNT),0) as w_60
          ,round(sum(F_70_CNT),0)          as w_70
      from e10_유동인구_flowpop
     group by
           substring(년월일,1,6)
          ,행정코드
          ,시군구명
          ,행정동명
          ,요일구분
          ,시간구분
-- )select * from t_1;
), t_2 as (
    select 년월
          ,행정코드
          ,시군구명
          ,행정동명
          ,b1.남여구분명
          ,b2.연령구분명
          ,요일구분
          ,시간구분
          ,case when b1.남여구분명='남자' and 연령구분명='10대이하' then  m_10
                when b1.남여구분명='남자' and 연령구분명='20대'    then  m_20
                when b1.남여구분명='남자' and 연령구분명='30대'    then  m_30
                when b1.남여구분명='남자' and 연령구분명='40대'    then  m_40
                when b1.남여구분명='남자' and 연령구분명='50대'    then  m_50
                when b1.남여구분명='남자' and 연령구분명='60대이상' then  m_60 + m_70
                when b1.남여구분명='여자' and 연령구분명='10대이하' then  w_10
                when b1.남여구분명='여자' and 연령구분명='20대'    then  w_20
                when b1.남여구분명='여자' and 연령구분명='30대'    then  w_30
                when b1.남여구분명='여자' and 연령구분명='40대'    then  w_40
                when b1.남여구분명='여자' and 연령구분명='50대'    then  w_50
                when b1.남여구분명='여자' and 연령구분명='60대이상' then  w_60 + w_70
                else 0
            end as CNT
      from t_1 a
     inner join a_1 b1
     inner join a_2 b2
-- )select * from t_2; /* [202401:74592] */
)   /* CNT_day|CNT_month 항목생성 */
    select a.년월
          ,a.시군구명
          ,a.행정동명 
          ,a.남여구분명
          ,a.연령구분명
          ,a.요일구분
          ,a.시간구분
          ,a.CNT
          ,round(a.CNT / b.days_in_month,1) as CNT_day
          ,round(a.CNT / c.cnt_month,1)     as CNT_month
      from t_2 a
      left join month_day_cnt b
        on a.년월 = b.년월
      left join month_cnt     c
        on substring(a.년월,1,4) = c.년

"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    log.info(f"{log_text[0]}.2.SQL 실행결과 가져오기 ")
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f"{log_text[0]}.3.테이블 DROP")
    cur.execute(f'DROP table IF EXISTS {config_info["p_table_name"]}')
    log.info(f"{log_text[0]}.4.결과저장처리")
    result = flime.f_data_save2(p_log=log, p_data_object=df, p_config_info=config_info)

    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 년월    VARCHAR(6) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 행정동명 VARCHAR(20) DEFAULT NULL NULL')
    cur.execute(f'CREATE INDEX {config_info["p_table_name"]}_년월_IDX ON {config_info["p_table_name"]} (년월,행정동명)')

    log.info(f"{log_text[0]}.[종료]{log_text[1]} [추출건수: {len(df)}] [처리시간:{fu.f_duration_time(p_s_time)}]")
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/23 20:27:27] 2.11.[종료][e11_유동인구_flowpop] [추출건수: 114912] [처리시간:0:00:07]
#[INFO   ][2024/07/25 15:58:20] 2.11.[종료][e11_유동인구_flowpop] [추출건수: 782208] [처리시간:0:01:01]

In [ ]:
###########################################################################
# [2.21][xlsx][1.유동인구][SX_AGE] ← [e11_유동인구_flowpop]
###########################################################################
# 1.원천 데이터 정보 : [e11_유동인구_flowpop]
# 2.타켓 데이터 정보 : [xlsx]
###########################################################################
# 1.로그 변수 선언
log_text = ["2.21","[1.유동인구][SX_AGE]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.파일 변수 선언
gubun                       = "xlsx" # 저장 형태 [csv, xlsx, db, txt, shp]
config_info["p_file_name"]  = "1.유동인구"
config_info["p_sheet_name"] = "SX_AGE"
config_info["p_file_path"]  = os.path.join(path_result,config_info["p_file_name"]); fu.mkdir_if_not_exists(config_info["p_file_path"])
# 3.SQL 데이터 추출 SQL
p_sql = f"""
-- [1.유동인구][SX_AGE]<-[e11_유동인구_flowpop]
with t_1 as (
    select a.년월
          ,substring(a.년월,1,4) as 년
          ,substring(a.년월,5,2) as 월
          ,a.시군구명
          ,a.행정동명
          ,a.남여구분명
          ,a.연령구분명
          -- ,a.요일구분
          -- ,a.시간구분
          ,sum(a.CNT)       as CNT
          ,sum(a.CNT_day)   as CNT_day
          ,sum(a.CNT_month) as CNT_month
          ,0                as 인구수
      from e11_유동인구_flowpop a
     group by
           a.년월
          ,a.시군구명
          ,a.행정동명
          ,a.남여구분명
          ,a.연령구분명
          -- ,a.요일구분
          -- ,a.시간구분
-- )select count(*) from t_1; /* [684] */
)
    select a.년월
          ,a.년
          ,a.월
          ,a.시군구명
          ,a.행정동명
          ,a.남여구분명
          ,a.연령구분명
          -- ,a.요일구분
          -- ,a.시간구분
          ,round(a.CNT,1)       as CNT
          ,round(b.CNT,1)       as CNT_전년동월
          ,round(a.CNT_day,1)   as CNT_day
          ,round(b.CNT_day,1)   as CNT_day_전년동월
          ,round(a.CNT_month,1) as CNT_month
          ,round(b.CNT_month,1) as CNT_month_전년동월
          ,a.인구수
          ,b.인구수 as 인구수_전년
      from      t_1 a
      left join t_1 b
        on DATE_FORMAT(DATE_SUB(STR_TO_DATE(CONCAT(a.년월,'01'), '%Y%m%d'), INTERVAL 12 MONTH), '%Y%m') = b.년월
       and a.시군구명  = b.시군구명
       and a.행정동명  = b.행정동명
       and a.남여구분명    = b.남여구분명
       and a.연령구분명    = b.연령구분명
       -- and a.요일구분  = b.요일구분
       -- and a.시간구분  = b.시간구분
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    # 4.1.Database 접속
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.SQL 실행결과 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')

    # 4.2.df 내의 열을 숫자형으로 변환
    list_col = ['CNT','CNT_전년동월','CNT_day','CNT_day_전년동월','CNT_month','CNT_month_전년동월']
    df = fu.convert_columns_to_numeric(df, list_col)

    log.info(f'{log_text[0]}.3.Excel 파일 저장하기 ')
    result = flime.f_data_to_excel(p_log = log, p_data_object = df, p_config_info = config_info)

    log.info(f'{log_text[0]}.[종료]{log_text[1]} [저장건수: {len(df)},({config_info["p_file_name"]} | {config_info["p_sheet_name"]})] [처리시간:{fu.f_duration_time(p_s_time)}]')
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/24 10:35:57] 2.21.[종료][1.유동인구][SX_AGE] [저장건수: 4656,(1.유동인구 | SX_AGE)] [처리시간:0:00:49]
#[INFO   ][2024/07/25 16:02:47] 2.21.[종료][1.유동인구][SX_AGE] [저장건수: 4656,(1.유동인구 | SX_AGE)] [처리시간:0:00:07]

In [ ]:
###########################################################################
# [2.22][xlsx][1.유동인구][DOW] ← [e11_유동인구_flowpop]
###########################################################################
# 1.원천 데이터 정보 : [e11_유동인구_flowpop]
# 2.타켓 데이터 정보 : [xlsx]
###########################################################################
# 1.로그 변수 선언
log_text = ["2.22","[1.유동인구][DOW]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.파일 변수 선언
gubun                       = "xlsx" # 저장 형태 [csv, xlsx, db, txt, shp]
config_info["p_file_name"]  = "1.유동인구"
config_info["p_sheet_name"] = "DOW"
config_info["p_file_path"]  = os.path.join(path_result,config_info["p_file_name"]); fu.mkdir_if_not_exists(config_info["p_file_path"])
# 3.SQL 데이터 추출 SQL
p_sql = f"""
-- [1.유동인구][DOW]<-[e11_유동인구_flowpop]
with t_1 as (
    select a.년월
          ,substring(a.년월,1,4) as 년
          ,substring(a.년월,5,2) as 월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.요일구분
          -- ,a.시간구분
          ,sum(a.CNT)       as CNT
          ,sum(a.CNT_day)   as CNT_day
          ,sum(a.CNT_month) as CNT_month
      from e11_유동인구_flowpop a
     group by
           a.년월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.요일구분
          -- ,a.시간구분
-- )select count(*) from t_1; /* [684] */
)
    select a.년월
          ,a.년
          ,a.월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.요일구분
          -- ,a.시간구분
          ,round(a.CNT,1)       as CNT
          ,round(b.CNT,1)       as CNT_전년동월
          ,round(a.CNT_day,1)   as CNT_day
          ,round(b.CNT_day,1)   as CNT_day_전년동월
          ,round(a.CNT_month,1) as CNT_month
          ,round(b.CNT_month,1) as CNT_month_전년동월
      from      t_1 a
      left join t_1 b
        on DATE_FORMAT(DATE_SUB(STR_TO_DATE(CONCAT(a.년월,'01'), '%Y%m%d'), INTERVAL 12 MONTH), '%Y%m') = b.년월
       and a.시군구명  = b.시군구명
       and a.행정동명  = b.행정동명
       -- and a.남여구분명 = b.남여구분명
       -- and a.연령구분명 = b.연령구분명
       and a.요일구분     = b.요일구분
       -- and a.시간구분  = b.시간구분
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    # 4.1.Database 접속
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.SQL 실행결과 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')

    # 4.2.df 내의 열을 숫자형으로 변환
    list_col = ['CNT','CNT_전년동월','CNT_day','CNT_day_전년동월','CNT_month','CNT_month_전년동월']
    df = fu.convert_columns_to_numeric(df, list_col)

    log.info(f'{log_text[0]}.3.Excel 파일 저장하기 ')
    result = flime.f_data_to_excel(p_log = log, p_data_object = df, p_config_info = config_info)

    log.info(f'{log_text[0]}.[종료]{log_text[1]} [저장건수: {len(df)},({config_info["p_file_name"]} | {config_info["p_sheet_name"]})] [처리시간:{fu.f_duration_time(p_s_time)}]')
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/25 16:04:38] 2.22.[종료][1.유동인구][DOW] [저장건수: 2716,(1.유동인구 | DOW)] [처리시간:0:00:08]

In [ ]:
###########################################################################
# [2.23][xlsx][1.유동인구][TIME] ← [e11_유동인구_flowpop]
###########################################################################
# 1.원천 데이터 정보 : [e11_유동인구_flowpop]
# 2.타켓 데이터 정보 : [xlsx]
###########################################################################
# 1.로그 변수 선언
log_text = ["2.23","[1.유동인구][TIME]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.파일 변수 선언
gubun                       = "xlsx" # 저장 형태 [csv, xlsx, db, txt, shp]
config_info["p_file_name"]  = "1.유동인구"
config_info["p_sheet_name"] = "TIME"
config_info["p_file_path"]  = os.path.join(path_result,config_info["p_file_name"]); fu.mkdir_if_not_exists(config_info["p_file_path"])
# 3.SQL 데이터 추출 SQL
p_sql = f"""
-- [1.유동인구][TIME]<-[e11_유동인구_flowpop]
with t_1 as (
    select a.년월
          ,substring(a.년월,1,4) as 년
          ,substring(a.년월,5,2) as 월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          -- ,a.요일구분
          ,a.시간구분
          ,sum(a.CNT)       as CNT
          ,sum(a.CNT_day)   as CNT_day
          ,sum(a.CNT_month) as CNT_month
      from e11_유동인구_flowpop a
     group by
           a.년월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          -- ,a.요일구분
          ,a.시간구분
-- )select count(*) from t_1; /* [684] */
)
    select a.년월
          ,a.년
          ,a.월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          -- ,a.요일구분
          ,a.시간구분
          ,round(a.CNT,1)       as CNT
          ,round(b.CNT,1)       as CNT_전년동월
          ,round(a.CNT_day,1)   as CNT_day
          ,round(b.CNT_day,1)   as CNT_day_전년동월
          ,round(a.CNT_month,1) as CNT_month
          ,round(b.CNT_month,1) as CNT_month_전년동월
      from      t_1 a
      left join t_1 b
        on DATE_FORMAT(DATE_SUB(STR_TO_DATE(CONCAT(a.년월,'01'), '%Y%m%d'), INTERVAL 12 MONTH), '%Y%m') = b.년월
       and a.시군구명  = b.시군구명
       and a.행정동명  = b.행정동명
       -- and a.남여구분명 = b.남여구분명
       -- and a.연령구분명 = b.연령구분명
       -- and a.요일구분  = b.요일구분
       and a.시간구분      = b.시간구분
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    # 4.1.Database 접속
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.SQL 실행결과 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')

    # 4.2.df 내의 열을 숫자형으로 변환
    list_col = ['CNT','CNT_전년동월','CNT_day','CNT_day_전년동월','CNT_month','CNT_month_전년동월']
    df = fu.convert_columns_to_numeric(df, list_col)

    log.info(f'{log_text[0]}.3.Excel 파일 저장하기 ')
    result = flime.f_data_to_excel(p_log = log, p_data_object = df, p_config_info = config_info)

    log.info(f'{log_text[0]}.[종료]{log_text[1]} [저장건수: {len(df)},({config_info["p_file_name"]} | {config_info["p_sheet_name"]})] [처리시간:{fu.f_duration_time(p_s_time)}]')
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/25 16:05:28] 2.23.[종료][1.유동인구][TIME] [저장건수: 9312,(1.유동인구 | TIME)] [처리시간:0:00:12]

In [ ]:
##############################################################################################
##############################################################################################
# [3.1][m10_xy_mapping100]:격자id 및 위경도 Mapping
# 1.원천 데이터 정보 : [d01_인구_100_공주시][d02_인구_100_읍면동][d04_skt]
# 2.타켓 데이터 정보 : [m10_xy_mapping100]
# 3.경계좌표계 변경 처리
#   - 데이터 좌표계     : WGS84 (EPSG:4326) : 경/위도
#   - 표준으로 변경 처리 : N/A
##############################################################################################
base_code = gpd.read_file('C:\\2024_project\\project\\A28_수원시_주차장수요공급분석\\P3_Result\\QGIS_DATA\\기초구역경계\\기초구역경계(시군구,행정동매핑).gpkg',driver='GPKG')
    
log_text = ["3.1","기초구역 경계면데이터"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()

#==========================================================================
p_sql = f"""
-- [d01_인구_100_공주시] gid격자정보 가져오기
with t_1 as (
    select
           a.기초구역번호
          ,a.SGG_NM
          ,b.HJD_NM
          ,b.geometry
      from base_code a
-- )select count(*) from t_1; /* [87541] */
)select * from t_1
"""

try:
    log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    # 2.100격자정보 가져오기
    '''
    log.info(f"{log_text[0]}.2.1.100격자정보 가져오기")
    df_100 = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.1.100격자정보 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')
    log.info(f"{log_text[0]}.2.2.[gdf_100]폴리곤정보 객체화")
    df_100["geometry"] = wkt.loads(df_100["geometry"])
    '''

    log.info(f"{log_text[0]}.4.1.매핑결과추출 : 필요항목만")
    mapping_after = base_code[['기초구역번호','SGG_NM','HJD_NM','geometry']]
    mapping_after.drop_duplicates(inplace=True)
    mapping_after = gpd.GeoDataFrame(mapping_after, geometry='geometry')
    # 4.2.매핑결과 DB저장처리
    log.info(f"{log_text[0]}.4.2.기초구역경계면 DB저장처리")
    config_info["p_gubun"]      = "db" # 저장 형태 [csv, xls, xlsx, db, txt, shp]
    config_info["p_table_name"] = "base_geometry"
    config_info["p_if_exists"]  = "replace" #[replace][append]
    log.info(f"{log_text[0]}.4.3.결과저장처리")
    result = flime.f_data_save2(p_log=log, p_data_object=mapping_after, p_config_info=config_info)

    log.info(f"{log_text[0]}.4.4.index생성처리")
#    cur.execute("ALTER TABLE {p_table_name} MODIFY COLUMN UTMK_X DOUBLE DEFAULT NULL NULL".format(**config_info))
#    cur.execute("ALTER TABLE {p_table_name} MODIFY COLUMN UTMK_Y DOUBLE DEFAULT NULL NULL".format(**config_info))
#    cur.execute("CREATE INDEX {p_table_name}UTMK_X ON {p_table_name} (UTMK_X,UTMK_Y)".format(**config_info))
#    cur.execute("ALTER TABLE {p_table_name} MODIFY COLUMN geometry TEXT DEFAULT NULL NULL".format(**config_info))
#    cur.execute("CREATE INDEX {p_table_name}geometry ON {p_table_name} (geometry)".format(**config_info))

    log.info(f"{log_text[0]}.[종료]{log_text[1]} [추출건수: {len(mapping_after)}] [처리시간:{fu.f_duration_time(p_s_time)}]")
except Exception as e:
    log.error(f'      file 예외가 발생했습니다.[{e}]')
finally :
    cur.close(); conn.close()
    fu.f_config_info_del(config_info)
    #df_100,base_code,df_skt_xy,gdf_skt_xy,mapping,mapping_after = None
    fu.f_del_objects([base_code,mapping,mapping_after])

#[INFO   ][2024/04/19 15:54:28] 3.1.[종료][base_geometry]:격자id 및 위경도 Mapping [추출건수: 507] [처리시간:0:08:58]